In [20]:
!pip install -q dill iterative-stratification
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


PyTorch 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [21]:
import os, csv, json, math, time, re
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

KAGGLE_INPUT = Path('/kaggle/input')
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def _count_imgs(d):
    try:
        return sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS)
    except (PermissionError, OSError):
        return 0

img_candidates = []
for p in KAGGLE_INPUT.rglob('*'):
    if not p.is_dir():
        continue
    n = _count_imgs(p)
    if n < 50:
        continue
    is_cropped = any(t in str(p).lower() for t in ('crop', 'cropped', 'bodyoutline'))
    img_candidates.append((p, n, is_cropped))

if not img_candidates:
    raise RuntimeError('No image folders found.')
img_candidates.sort(key=lambda t: (not t[2], -t[1]))
CROPPED_DIR  = next((c[0] for c in img_candidates if c[2]), None)
ORIGINAL_DIR = next((c[0] for c in img_candidates if not c[2]), None)
IMAGES_DIR = CROPPED_DIR if CROPPED_DIR else ORIGINAL_DIR
print(f'IMAGES_DIR: {IMAGES_DIR}')

BASE_DIR = Path('/kaggle/working/vkr')
CHECKPOINTS_DIR = BASE_DIR / 'checkpoints'
RESULTS_DIR = Path('/kaggle/working/results')
for d in [BASE_DIR, CHECKPOINTS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


IMAGES_DIR: /kaggle/input/datasets/serafimsh/vkr-drawings-cropped/crops


CSV


In [22]:
report_path, results_path, crop_meta_path = None, None, None
for f in KAGGLE_INPUT.rglob('*.csv'):
    nl = f.name.lower()
    if nl == 'crop_meta.csv':
        crop_meta_path = f
    elif 'report' in nl and 'filtered' not in nl and report_path is None:
        report_path = f
    elif 'result' in nl and results_path is None:
        results_path = f
if report_path is None or results_path is None:
    raise RuntimeError('CSV not found')
print(f'report:    {report_path}')
print(f'results:   {results_path}')
print(f'crop_meta: {crop_meta_path}')


report:    /kaggle/input/datasets/serafimsh/vkr-drawings/report-125327_kriterii-testa-risunok-cheloveka-obnovlennaya__202509031100.csv
results:   /kaggle/input/datasets/serafimsh/vkr-drawings/results_resnet152_f1_BodyOutlineYolo_final.csv
crop_meta: /kaggle/input/datasets/serafimsh/vkr-drawings-cropped/crop_meta.csv


Парсинг 


In [24]:
def parse_report(path):
    with open(path, 'r', encoding='utf-8') as f:
        rows = list(csv.reader(f))
    header_page, header_q, header_opts = rows[0], rows[1], rows[2]
    data_rows = rows[4:]
    col_names = []
    for i, (p, q, o) in enumerate(zip(header_page, header_q, header_opts)):
        parts = [x.strip() for x in [p, q, o] if x.strip()]
        col_names.append(' | '.join(parts) if parts else f'col_{i}')
    records = []
    for row in data_rows:
        if not row or not row[0].strip(): continue
        rec = {col_names[i]: val.strip() for i, val in enumerate(row) if i < len(col_names)}
        records.append(rec)
    return col_names, records

def parse_criteria(path):
    crit = {}
    with open(path, 'r', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row['column'] != 'mean':
                crit[row['column']] = {'f1_macro': float(row['f1_macro']),
                                        'f1_micro': float(row['f1_micro']),
                                        'f1_weighted': float(row['f1_weighted'])}
    return crit

col_names, records = parse_report(report_path)
criteria = parse_criteria(results_path)
code_col = col_names[9]
F1_THRESHOLD = 0.8
TARGET_CRITERIA = {n for n, m in criteria.items() if m['f1_macro'] < F1_THRESHOLD}
print(f'Records: {len(records)}, criteria: {len(criteria)}, target: {len(TARGET_CRITERIA)}')


Records: 8582, criteria: 117, target: 24


In [ ]:
import re as _re
with open(report_path, 'r', encoding='utf-8') as _f:
    _raw = list(csv.reader(_f))
_row1, _row2 = _raw[1], _raw[2]

def _clean_q(q):
    q = _re.sub(r'^\d+[\.\s]*', '', q)
    q = _re.sub(r'\s*\([^)]*(?:выбор|ответ)[^)]*\)\s*$', '', q, flags=_re.IGNORECASE)
    q = q.replace('\xa0', ' ')
    return _re.sub(r'\s+', ' ', q).strip().lower()

_cur_q, _cur_type = '', ''
col_struct = []
for i in range(len(_row1)):
    r1, r2 = _row1[i].strip(), _row2[i].strip()
    if r1:
        _cur_q = r1
        m = _re.search(r'\(([^)]*)\)', r1)
        _cur_type = m.group(1).lower() if m else ''
    col_struct.append({'idx': i, 'q_clean': _clean_q(_cur_q), 'option': r2,
        'opt_lower': r2.strip().lower(), 'is_multi': 'множественный' in _cur_type})

def _parse_suffix(name):
    m = _re.search(r'\.(\d+)$', name)
    return (name[:m.start()], int(m.group(1))) if m else (name, 0)

criteria_col_map, unmapped = {}, []
for crit_name in criteria.keys():
    base_name, suffix_num = _parse_suffix(crit_name)


    
    parts = base_name.split(' | ')
    if len(parts) == 2:
        feat, var = parts[0].strip().lower(), parts[1].strip().lower()
        matches = [cs['idx'] for cs in col_struct if cs['opt_lower'] == var and feat in cs['q_clean']]
        if not matches:
            ws = [w for w in feat.split() if len(w) > 2]
            matches = [cs['idx'] for cs in col_struct if cs['opt_lower'] == var and ws and all(w in cs['q_clean'] for w in ws[:2])]
        if matches:
            criteria_col_map[crit_name] = ('binary', matches[suffix_num] if suffix_num < len(matches) else matches[0])
        else:
            unmapped.append(crit_name)
    else:
        feat = parts[0].strip().lower()
        occ = [cs['idx'] for cs in col_struct if not cs['is_multi'] and not cs['option'] and feat == cs['q_clean']]
        if not occ:
            occ = [cs['idx'] for cs in col_struct if not cs['is_multi'] and not cs['option'] and feat in cs['q_clean']]
        if occ:
            criteria_col_map[crit_name] = ('multiclass', occ[suffix_num] if suffix_num < len(occ) else occ[-1])
        else:
            unmapped.append(crit_name)

criteria_configs = []
for name, m in criteria.items():
    if name not in criteria_col_map: continue
    task_type, col_idx = criteria_col_map[name]
    criteria_configs.append({'name': name, 'task_info': {'type': task_type, 'col_idx': col_idx},
        'f1_macro_old': m['f1_macro'], 'f1_micro_old': m['f1_micro'], 'f1_weighted_old': m['f1_weighted'],
        'priority': 'high' if name in TARGET_CRITERIA else 'normal'})
criteria_configs.sort(key=lambda x: x['f1_macro_old'])
print(f'Mapped: {len(criteria_configs)}/{len(criteria)}')


Mapped: 117/117


In [ ]:
BBOX_DIM = 12

def normalize(s):
    return s.strip().replace(' ', '').lower()

def stem_key(filename):
    return normalize(Path(filename).stem)

def compute_bbox_feats(x1, y1, x2, y2, W, H, src='yolo'):
    if W <= 0 or H <= 0: return [0.0] * BBOX_DIM
    bw = max(x2 - x1, 1.0); bh = max(y2 - y1, 1.0)


    #
    cx = (x1 + x2) / 2; cy = (y1 + y2) / 2
    rw = bw / W; rh = bh / H
    aspect = bh / bw
    touches_top    = 1.0 if y1 < 0.05 * H else 0.0
    touches_bottom = 1.0 if y2 > 0.95 * H else 0.0
    touches_left   = 1.0 if x1 < 0.05 * W else 0.0
    touches_right  = 1.0 if x2 > 0.95 * W else 0.0
    diag = (W**2 + H**2) ** 0.5
    min_corner = min(((cx - cxx)**2 + (cy - cyy)**2) ** 0.5
                     for cxx in (0, W) for cyy in (0, H)) / diag
    is_yolo = 1.0 if str(src).startswith('yolo') else 0.0
    return [rw, rh, rw * rh, cx / W, cy / H, min(aspect, 5.0),
            touches_top, touches_bottom, touches_left, touches_right,
            min_corner, is_yolo]

bbox_feats_by_code = {}
if crop_meta_path is not None:
    with open(crop_meta_path, encoding='utf-8') as f:
        for r in csv.DictReader(f):
            try:
                src = r.get('bbox_source', '')
                if src.startswith('error'): continue
                feats = compute_bbox_feats(float(r['x1']), float(r['y1']),
                    float(r['x2']), float(r['y2']),
                    float(r['orig_w']), float(r['orig_h']), src)
                bbox_feats_by_code[stem_key(r['src_file'])] = feats
            except (KeyError, ValueError): continue
    print(f'bbox для {len(bbox_feats_by_code)} кодов')


bbox для 8175 кодов


In [27]:
available_images = {}
for f in IMAGES_DIR.iterdir():
    if (f.is_file() or f.is_symlink()) and f.suffix.lower() in IMAGE_EXTENSIONS:
        available_images[normalize(f.stem)] = str(f)
print(f'Images: {len(available_images)}')


Images: 8175


 Majority- vote 


In [29]:
records_by_code = defaultdict(list)
for rec in records:
    code = rec.get(code_col, '').strip()
    if not code or normalize(code) not in available_images: continue
    records_by_code[code].append(rec)

def majority_label(values, task_type):
    if task_type == 'binary':
        pos = sum(1 for v in values if v)
        return 1 if pos * 2 >= len(values) and pos > 0 else 0
    non_empty = [v for v in values if str(v).strip()]
    if not non_empty: return ''
    ctr = Counter(non_empty)
    top_cnt = ctr.most_common(1)[0][1]
    top_vals = [v for v, c in ctr.items() if c == top_cnt]
    if len(top_vals) == 1: return top_vals[0]
    for v in non_empty:
        if v in top_vals: return v
    return non_empty[0]

dataset = []
for code, recs in records_by_code.items():
    labels = {}
    for cfg in criteria_configs:
        n = cfg['name']; ci = cfg['task_info']['col_idx']; col = col_names[ci]
        vals = [r.get(col, '').strip() for r in recs]
        labels[n] = majority_label(vals, cfg['task_info']['type'])
    feats = bbox_feats_by_code.get(normalize(code), [0.0] * BBOX_DIM)
    dataset.append({'code': code, 'image_path': available_images[normalize(code)],
        'labels': labels, 'bbox_feats': feats})
print(f'Dataset: {len(dataset)}')

pos_counts = {}
for cfg in criteria_configs:
    n = cfg['name']
    if cfg['task_info']['type'] == 'binary':
        pos_counts[n] = sum(1 for d in dataset if d['labels'].get(n) == 1)
    else:
        pos_counts[n] = dict(Counter(d['labels'].get(n, '') for d in dataset))

insufficient = set()
for cfg in criteria_configs:
    n = cfg['name']; pc = pos_counts[n]
    if cfg['task_info']['type'] == 'binary':
        if pc < 10: insufficient.add(n)
    else:
        non_empty = {k: v for k, v in pc.items() if k}
        if len(non_empty) < 2 or sum(non_empty.values()) < 20:
            insufficient.add(n)


Dataset: 8175


In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
binary_names = [c['name'] for c in criteria_configs if c['task_info']['type'] == 'binary']
multi_names  = [c['name'] for c in criteria_configs if c['task_info']['type'] == 'multiclass']
target_names = {c['name'] for c in criteria_configs if c['priority'] == 'high'}
N = len(dataset)
rare_cols = []
for bn in binary_names:
    if bn not in target_names: continue
    col = [1 if dataset[i]['labels'].get(bn) == 1 else 0 for i in range(N)]
    s = sum(col)
    if 1 < s < N: rare_cols.append(col)
for mn in multi_names:
    if mn not in target_names: continue
    ctr = Counter(dataset[i]['labels'].get(mn, '') for i in range(N))
    for cls, cnt in ctr.items():
        if cls and 1 < cnt < N and cnt / N < 0.3:
            rare_cols.append([1 if dataset[i]['labels'].get(mn) == cls else 0 for i in range(N)])

if rare_cols:
    Y = np.array(rare_cols, dtype=int).T; X = np.arange(N).reshape(-1, 1)
    mss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
    train_idx, rest_idx = next(mss1.split(X, Y))
    mss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
    v_sel, t_sel = next(mss2.split(rest_idx.reshape(-1, 1), Y[rest_idx]))
    val_idx, test_idx = rest_idx[v_sel], rest_idx[t_sel]
else:
    rng = np.random.default_rng(42); perm = rng.permutation(N)
    n_tr, n_va = int(N * 0.70), int(N * 0.15)
    train_idx, val_idx, test_idx = perm[:n_tr], perm[n_tr:n_tr+n_va], perm[n_tr+n_va:]

train_data=[dataset[i] for i in train_idx]
val_data   = [dataset[i] for i in val_idx]
test_data  = [dataset[i] for i in test_idx]
print(f'Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}')

_train_feats = np.array([d['bbox_feats'] for d in train_data], dtype=np.float32)
BBOX_MEAN = _train_feats.mean(axis=0)
BBOX_STD  = _train_feats.std(axis=0) + 1e-6

label_encoders = {}
for cfg in criteria_configs:
    if cfg['task_info']['type'] != 'multiclass': continue
    name = cfg['name']
    vals = sorted({str(item['labels'].get(name, '')).strip() for item in train_data
                   if str(item['labels'].get(name, '')).strip()})
    enc = {'': 0}
    for i, v in enumerate(vals, 1): enc[v] = i
    label_encoders[name] = enc

train_pos = {}
for cfg in criteria_configs:
    n = cfg['name']
    if cfg['task_info']['type'] == 'binary':
        train_pos[n] = sum(1 for d in train_data if d['labels'].get(n) == 1)
    else:
        train_pos[n] = Counter(d['labels'].get(n, '') for d in train_data)


Train: 5722  Val: 1226  Test: 1227


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F_t
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, ImageFile
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score as sklearn_f1

ImageFile.LOAD_TRUNCATED_IMAGES = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMG_SIZE       = 320
BATCH_SIZE     = 32        
LR             = 1e-4      
WEIGHT_DECAY   = 5e-4

DROPOUT        = 0.3
EPOCHS         = 50        
WARMUP_EPOCHS  = 3
FOCAL_GAMMA_BIN = 0.0     
LABEL_SMOOTH    = 0.05
USE_AMP         = False    
SEED            = 42

print(f'Device {DEVICE} | IMG {IMG_SIZE} | BS {BATCH_SIZE} | Epochs {EPOCHS}')

def sanitize_column(col):
    return re.sub(r'\W|^(?=\d)', '_', col)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0, ignore_index=-100):
        super().__init__()
        if alpha is not None: self.register_buffer('alpha', alpha)
        else: self.alpha = None
        self.gamma = gamma; self.label_smoothing = label_smoothing; self.ignore_index = ignore_index
    def forward(self, logits, targets):
        ce = F_t.cross_entropy(logits, targets, weight=self.alpha,
            label_smoothing=self.label_smoothing, ignore_index=self.ignore_index, reduction='none')
        if self.gamma == 0:
            return ce.mean() if (targets != self.ignore_index).any() else logits.sum() * 0.0
        pt = torch.exp(-ce); loss = ((1 - pt) ** self.gamma) * ce
        mask = targets != self.ignore_index

        return loss[mask].mean() if mask.any() else logits.sum() * 0.0

BBOX_MEAN_T = torch.tensor(BBOX_MEAN, dtype=torch.float32)
BBOX_STD_T  = torch.tensor(BBOX_STD,  dtype=torch.float32)

class TwoStreamDataset(Dataset):
    def __init__(self, data, configs, encoders, transform, mean, std):
        self.data = data; self.configs = configs; self.encoders = encoders
        self.transform = transform; self.mean = mean; self.std = std
        self._bad = set()
    def __len__(self): return len(self.data)
    def _labels(self, item):
        out = {}
        for cfg in self.configs:
            n = cfg['name']; raw = item['labels'].get(n, '')
            if cfg['task_info']['type'] == 'binary':
                out[n] = torch.tensor(int(raw) if raw else 0, dtype=torch.long)
            else:
                v = str(raw).strip()
                out[n] = torch.tensor(self.encoders.get(n, {}).get(v, 0), dtype=torch.long)
        return out
    def __getitem__(self, idx):
        for shift in range(min(10, len(self.data))):
            i =(idx + shift) % len(self.data)
            item = self.data[i]
            try:
                img = Image.open(item['image_path']).convert('RGB')
                if self.transform: img = self.transform(img)
                bbox = torch.tensor(item['bbox_feats'], dtype=torch.float32)
                bbox = (bbox - self.mean) / self.std
                return img, bbox, self._labels(item)
            except Exception:
                if item['image_path'] not in self._bad:
                    self._bad.add(item['image_path'])
                    print(f'[WARN] Bad image: {item["image_path"]}')
        raise RuntimeError('10 bad images in a row')


class ResidualBlock(nn.Module):

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
        else:
            self.shortcut = nn.Identity()
    def forward(self, x):
        out = F_t.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F_t.relu(out, inplace=True)

class DrawingNet(nn.Module):
    IMG_FEAT_DIM = 256 + 384
    BBOX_FEAT_DIM = 32
    PROJ_DIM = 256
    def __init__(self, num_classes_dict, target_safe_names, nontarget_safe_names,
                 bbox_in_dim=BBOX_DIM, target_cols=None, label_encoders=None,
                 image_transform=None, safe_col_map=None, dropout=0.3, device='cpu'):
        super().__init__()
        # 11

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1))
        # 2
        self.stage2 = nn.Sequential(
            ResidualBlock(32, 64),
            ResidualBlock(64, 64),
            nn.MaxPool2d(2))
        self.stage3 = nn.Sequential(
            ResidualBlock(64, 128),
            ResidualBlock(128, 128),
            nn.MaxPool2d(2))
        self.stage4 = nn.Sequential(
            ResidualBlock(128, 256),
            ResidualBlock(256, 256),
            nn.MaxPool2d(2))

        self.stage5 = nn.Sequential(
            ResidualBlock(256, 384))
        self.pool = nn.AdaptiveAvgPool2d(1)


        self.bbox_mlp = nn.Sequential(
            nn.Linear(bbox_in_dim, 32), nn.ReLU(inplace=True), nn.Dropout(0.1),
            nn.Linear(32, self.BBOX_FEAT_DIM), nn.ReLU(inplace=True))


        ##### 
        self.target_proj = nn.Sequential(
            nn.Linear(self.IMG_FEAT_DIM + self.BBOX_FEAT_DIM, self.PROJ_DIM),
            nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.nontarget_proj = nn.Sequential(
            nn.Linear(self.IMG_FEAT_DIM + self.BBOX_FEAT_DIM, self.PROJ_DIM),
            nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.target_safe = set(target_safe_names)
        self.nontarget_safe = set(nontarget_safe_names)
        self.heads = nn.ModuleDict({col: nn.Linear(self.PROJ_DIM, n)
            for col, n in num_classes_dict.items()})
        self.target_cols = target_cols; self.label_encoders = label_encoders
        self.image_transform = image_transform; self.safe_col_map = safe_col_map
        self.device = device
        self.history = {'train_loss': [], 'val_f1_all': []}
        self._init_weights()
        self.to(device)
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
    def backbone_parameters(self):
        for m in [self.stem, self.stage2, self.stage3, self.stage4, self.stage5]:
            yield from m.parameters()
    def head_parameters(self):
        yield from self.bbox_mlp.parameters()
        yield from self.target_proj.parameters()
        yield from self.nontarget_proj.parameters()
        yield from self.heads.parameters()
    def forward(self, img, bbox):
        x = self.stem(img)
        x = self.stage2(x)
        x = self.stage3(x)
        f4 = self.stage4(x)
        f5 = self.stage5(f4)
        img_feat = torch.cat([self.pool(f4).flatten(1), self.pool(f5).flatten(1)], dim=1)
        bbox_feat = self.bbox_mlp(bbox)
        feat = torch.cat([img_feat, bbox_feat], dim=1)
        t_feat = self.target_proj(feat); n_feat = self.nontarget_proj(feat)
        return {col: head(t_feat if col in self.target_safe else n_feat)
                for col, head in self.heads.items()}
    def update_history(self, **kw):
        for k, v in kw.items(): self.history.setdefault(k, []).append(v)


#ntcn
_test_num_classes = {f'col_{i}': 2 for i in range(117)}
_test_target = {f'col_{i}' for i in range(24)}
_test_nontarget = {f'col_{i}' for i in range(24, 117)}
_m = DrawingNet(_test_num_classes, _test_target, _test_nontarget)
_n = sum(p.numel() for p in _m.parameters())
print(f'DrawingNet параметров: {_n/1e6:.2f}M')
del _m


Device cuda | IMG 320 | BS 32 | Epochs 50
DrawingNet параметров: 5.48M


Обучение

In [ ]:
IMAGENET_MEAN= [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

image_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomAffine(degrees=6, translate=(0.05, 0.05), scale=(0.90, 1.10)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),

    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.10), ratio=(0.5, 2.0))])
val_tf = image_transform

target_cols  = [c['name'] for c in criteria_configs]
safe_col_map = {c: sanitize_column(c) for c in target_cols}
num_classes_safe = {}
for cfg in criteria_configs:
    n = cfg['name']; sn = safe_col_map[n]
    if cfg['task_info']['type'] == 'binary':
        num_classes_safe[sn] = 2
    else:
        num_classes_safe[sn] = max(len(label_encoders.get(n, {})), 2)
target_safe_names    ={safe_col_map[c['name']] for c in criteria_configs if c['priority'] == 'high'}
nontarget_safe_names = {safe_col_map[c['name']] for c in criteria_configs if c['priority'] != 'high'}
target_col_names     = [c['name'] for c in criteria_configs if c['priority'] == 'high']
nontarget_col_names  = [c['name'] for c in criteria_configs if c['priority'] == 'normal']

sk_label_encoders = {}
for cfg in criteria_configs:
    n = cfg['name']; le = LabelEncoder()
    if cfg['task_info']['type'] == 'binary':
        le.fit(['0', '1'])
    else:
        enc = label_encoders.get(n, {})
        classes = sorted(enc.keys(), key=lambda k: enc[k]) or ['']
        le.fit(classes)
    sk_label_encoders[n] = le


def build_sampler_weights(train_data):
    N_tr = len(train_data); w = np.ones(N_tr, dtype=np.float32)
    for cfg in criteria_configs:
        if cfg['priority'] != 'high': continue
        name = cfg['name']
        if cfg['task_info']['type'] == 'binary':
            pos_idx = [i for i, d in enumerate(train_data) if d['labels'].get(name) == 1]
            if 0 < len(pos_idx) < 150:
                boost = min(3.0, 60.0 / max(len(pos_idx), 1))
                for i in pos_idx: w[i] += boost
        else:
            ctr = Counter(d['labels'].get(name, '') for d in train_data)
            for cls, cnt in ctr.items():
                if not cls or cnt == 0 or cnt >= 200: continue
                boost = min(2.0, 50.0 / cnt)
                for i, d in enumerate(train_data):
                    if d['labels'].get(name) == cls: w[i] += boost
    return np.clip(w, 1.0, 2.5)

def build_loss_fns():
    losses = {}
    for cfg in criteria_configs:
        name = cfg['name']
        if cfg['task_info']['type'] == 'binary':
            pos = train_pos[name]; neg = len(train_data) - pos
            w_pos = min((neg / max(pos, 1)) ** 0.5, 10.0)
            #print('--------')
            alpha = torch.tensor([1.0, max(w_pos, 1.0)], dtype=torch.float32, device=DEVICE)
            losses[name] =FocalLoss(alpha=alpha, gamma=FOCAL_GAMMA_BIN, label_smoothing=0.0)
        else:
            enc = label_encoders.get(name, {'': 0}); ctr = train_pos[name]
            
            keys = sorted(enc.keys(), key=lambda k: enc[k])
            counts = np.array([max(ctr.get(cls, 0), 1) for cls in keys], dtype=np.float64)
            mx = float( counts.max())

            alpha = np.minimum(np.sqrt(mx / counts), 10.0)
            alpha_t = torch.tensor(alpha, dtype=torch.float32, device=DEVICE)
            losses[name] = FocalLoss(alpha=alpha_t, gamma=0.0,
                label_smoothing=LABEL_SMOOTH, ignore_index=-100)
    return losses

sample_weights = build_sampler_weights(train_data)


print(f'Sampler: min={sample_weights.min():.2f} mean={sample_weights.mean():.2f} max={sample_weights.max():.2f}')


Sampler: min=1.00 mean=1.25 max=2.50


In [ ]:
import dill

def cosine_warmup_lr(epoch, total, warmup):
    if epoch < warmup: return (epoch + 1) / max(warmup, 1)
    t = (epoch - warmup) / max(total - warmup, 1)
    return 0.5 * (1 + math.cos(math.pi * t))

torch.manual_seed(SEED); np.random.seed(SEED)

train_ds = TwoStreamDataset(train_data, criteria_configs, label_encoders, train_tf, BBOX_MEAN_T, BBOX_STD_T)
val_ds   = TwoStreamDataset(val_data,   criteria_configs, label_encoders, val_tf,   BBOX_MEAN_T, BBOX_STD_T)

sampler = WeightedRandomSampler(weights=torch.from_numpy(sample_weights).double(),
                                 num_samples=len(train_data), replacement=True)
train_loader =DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = DrawingNet(
    num_classes_dict=num_classes_safe,
    target_safe_names=target_safe_names, nontarget_safe_names=nontarget_safe_names,
    bbox_in_dim=BBOX_DIM,
    target_cols=target_cols, label_encoders=sk_label_encoders,
    image_transform=image_transform, safe_col_map=safe_col_map,
    dropout=DROPOUT, device=str(DEVICE))

loss_fns = build_loss_fns()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and torch.cuda.is_available())
LOSS_NORM = float(max(len(criteria_configs), 1))

best_f1=0.0
best_state_path = CHECKPOINTS_DIR / f'drawingnet_seed{SEED}.pth'
patience = 8; no_improve = 0; MIN_EPOCHS = 25

for epoch in range(EPOCHS):
    lr_mult = cosine_warmup_lr(epoch, EPOCHS,WARMUP_EPOCHS)
    for pg in opt.param_groups:
        base = pg.get('_base_lr', pg['lr']); pg['_base_lr'] = base; pg['lr'] = base * lr_mult
    model.train()
    train_loss = 0.0; n_b = 0; t0 = time.time()
    for imgs, bboxes, lbl in train_loader:
        imgs = imgs.to(DEVICE, non_blocking=True); bboxes = bboxes.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP and torch.cuda.is_available()):
            out = model(imgs, bboxes)
            total = torch.tensor(0.0, device=DEVICE)
            for cfg in criteria_configs:
                n = cfg['name']; sn = safe_col_map[n]
                tg = lbl[n].long().to(DEVICE, non_blocking=True)
                total = total + loss_fns[n](out[sn], tg)
            loss = total / LOSS_NORM
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        train_loss += loss.item(); n_b += 1
    avg_train = train_loss / max(n_b, 1)
    model.eval()
    all_p = {n: [] for n in target_cols}; all_t = {n: [] for n in target_cols}
    with torch.no_grad():
        for imgs, bboxes, lbl in val_loader:
            imgs = imgs.to(DEVICE, non_blocking=True); bboxes = bboxes.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=USE_AMP and torch.cuda.is_available()):
                out = model(imgs, bboxes)
            for cfg in criteria_configs:
                n = cfg['name']; sn = safe_col_map[n]
                all_p[n].extend(out[sn].argmax(dim=-1).cpu().numpy().tolist())
                all_t[n].extend(lbl[n].long().numpy().tolist())
    f1s_macro    = {n: sklearn_f1(all_t[n], all_p[n], average='macro', zero_division=0) for n in target_cols}
    f1s_micro    = {n: sklearn_f1(all_t[n], all_p[n], average='micro', zero_division=0) for n in target_cols}
    f1s_weighted = {n: sklearn_f1(all_t[n], all_p[n], average='weighted', zero_division=0) for n in target_cols}
    f1_all_macro    = float(np.mean([f1s_macro[n] for n in target_cols]))
    
    f1_all_micro    = float(np.mean([f1s_micro[n] for n in target_cols]))
    f1_all_weighted = float(np.mean([f1s_weighted[n] for n in target_cols]))
    dt = time.time() - t0; lr_now = opt.param_groups[0]['lr']
    print(f'Ep {epoch+1:2d}/{EPOCHS} | {dt:5.1f}s | Loss {avg_train:.4f} | '
          f'F1 macro {f1_all_macro:.4f} | F1 micro {f1_all_micro:.4f} | F1 weighted {f1_all_weighted:.4f} | LR {lr_now:.2e}')
    model.update_history(train_loss=avg_train, val_f1_all=f1_all_weighted)
    if f1_all_weighted > best_f1:
        best_f1 = f1_all_weighted; no_improve = 0
        cpu_model = model.to('cpu'); cpu_model.device = 'cpu'
        torch.save(cpu_model.state_dict(), best_state_path)
        model.to(DEVICE); model.device = str(DEVICE)
        print(f'  [SAVE] best F1_weighted: {best_f1:.4f}')
    else:
        no_improve += 1
        if no_improve >= patience and epoch >= MIN_EPOCHS:
            print('  Early stopping')
            break

print(f'\n===  best val F1_weighted: {best_f1:.4f} ===')


Ep  1/50 |  61.4s | Loss 4.1650 | F1 macro 0.3947 | F1 micro 0.6765 | F1 weighted 0.7246 | LR 3.33e-05
  [SAVE] best F1_weighted: 0.7246
Ep  2/50 |  60.7s | Loss 1.1200 | F1 macro 0.3855 | F1 micro 0.6572 | F1 weighted 0.6891 | LR 6.67e-05
Ep  3/50 |  61.1s | Loss 1.0822 | F1 macro 0.3835 | F1 micro 0.6831 | F1 weighted 0.6775 | LR 1.00e-04
Ep  4/50 |  64.6s | Loss 1.0638 | F1 macro 0.3871 | F1 micro 0.7164 | F1 weighted 0.7096 | LR 1.00e-04
Ep  5/50 |  61.0s | Loss 1.0384 | F1 macro 0.4160 | F1 micro 0.7782 | F1 weighted 0.7620 | LR 9.99e-05
  [SAVE] best F1_weighted: 0.7620
Ep  6/50 |  61.3s | Loss 1.0103 | F1 macro 0.4389 | F1 micro 0.8003 | F1 weighted 0.7860 | LR 9.96e-05
  [SAVE] best F1_weighted: 0.7860
Ep  7/50 |  60.7s | Loss 0.9930 | F1 macro 0.4398 | F1 micro 0.8192 | F1 weighted 0.7988 | LR 9.90e-05
  [SAVE] best F1_weighted: 0.7988
Ep  8/50 |  60.5s | Loss 0.9740 | F1 macro 0.4520 | F1 micro 0.8404 | F1 weighted 0.8219 | LR 9.82e-05
  [SAVE] best F1_weighted: 0.8219
Ep  9/

Тест и отчет

In [37]:
model.load_state_dict(torch.load(best_state_path, map_location=DEVICE))
model.to(DEVICE); model.device = str(DEVICE); model.eval()

test_ds = TwoStreamDataset(test_data, criteria_configs, label_encoders, val_tf, BBOX_MEAN_T, BBOX_STD_T)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

tp = {n: [] for n in target_cols}; tt = {n: [] for n in target_cols}
with torch.no_grad():
    for imgs, bboxes, lbl in test_loader:
        imgs = imgs.to(DEVICE, non_blocking=True); bboxes = bboxes.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP and torch.cuda.is_available()):
            out = model(imgs, bboxes)
        for cfg in criteria_configs:
            n = cfg['name']; sn = safe_col_map[n]
            tp[n].extend(out[sn].argmax(dim=-1).cpu().numpy().tolist())
            tt[n].extend(lbl[n].long().numpy().tolist())

results = []
for cfg in criteria_configs:
    n = cfg['name']; y, p = tt[n], tp[n]
    f1m = sklearn_f1(y, p, average='macro', zero_division=0)
    f1mi = sklearn_f1(y, p, average='micro', zero_division=0)
    f1w = sklearn_f1(y, p, average='weighted', zero_division=0)
    if cfg['task_info']['type'] == 'binary': pc = int(train_pos[n])
    else: pc = int(sum(v for k, v in train_pos[n].items() if k))
    results.append({'criterion': n, 'task_type': cfg['task_info']['type'], 'priority': cfg['priority'],
        'pos_train': pc, 'insufficient_data': n in insufficient,
        'new_f1_macro': f1m, 'new_f1_micro': f1mi, 'new_f1_weighted': f1w})

import pandas as pd
df = pd.DataFrame(results)
print(f'\n=== RESULTS ==========')
print(f'F1_macro mean:    {df["new_f1_macro"].mean():.4f}')
print(f'F1_micro mean:    {df["new_f1_micro"].mean():.4f}')
print(f'F1_weighted mean: {df["new_f1_weighted"].mean():.4f}')
print(f'\nПо приоритетам:')
for prio in ['high', 'normal']:
    sub = df[df['priority'] == prio]
    print(f'  {prio:8s} ({len(sub):3d}): macro={sub["new_f1_macro"].mean():.4f} '
          f'micro={sub["new_f1_micro"].mean():.4f} weighted={sub["new_f1_weighted"].mean():.4f}')

df.to_csv(RESULTS_DIR / 'results_drawingnet_v4_detailed.csv', index=False, encoding='utf-8-sig')
compat = pd.DataFrame({'column': df['criterion'], 'f1_macro': df['new_f1_macro'],
    'f1_micro': df['new_f1_micro'], 'f1_weighted': df['new_f1_weighted']})
compat.loc[len(compat)] = {'column': 'mean', 'f1_macro': df['new_f1_macro'].mean(),
    'f1_micro': df['new_f1_micro'].mean(), 'f1_weighted': df['new_f1_weighted'].mean()}
compat.to_csv(RESULTS_DIR / 'results_drawingnet_v4.csv', index=False, encoding='utf-8-sig')
print(f'\nSaved to {RESULTS_DIR}')



=== RESULTS ==========
F1_macro mean:    0.4635
F1_micro mean:    0.8998
F1_weighted mean: 0.8668

По приоритетам:
  high     ( 24): macro=0.3852 micro=0.9259 weighted=0.9071
  normal   ( 93): macro=0.4838 micro=0.8931 weighted=0.8564

Saved to /kaggle/working/results
